<div style="text-align:right">
<img src="https://raw.githubusercontent.com/cayetanoguerra/redes-neuronales/main/notebooks/imgs/logo-curso.jpg" width="30%" style="max-width:100%; padding-right:20px;">
</div>

**Fundamentos de la Inteligencia Artificial y Programación de APIs para LLMs**

# **09. API: cómo hablan los programas con los modelos**

<div class="colab-badge">
<a href="https://colab.research.google.com/github/cayetanoguerra/redes-neuronales/blob/main/notebooks/09-api.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"></a>
<br><em>Para trabajar con tu propia copia, ábrelo en Colab y elige <b>Archivo → Guardar una copia en Drive</b>.</em>
</div>

En el notebook anterior conversamos con un modelo en la ventana de chat de LM Studio. Al final encendimos su servidor local y, con una celda de código, le enviamos una pregunta desde Python. Prometimos explicar aquella celda con calma, y ha llegado el momento.

Cuando un programa quiere usar un modelo de lenguaje no abre una ventana de chat: le envía una **petición** a través de una **API** y recibe una **respuesta**. Así funcionan las aplicaciones que integran inteligencia artificial, desde un asistente de correo hasta una plataforma educativa, y así trabajaremos en el resto del curso.

En este notebook veremos qué es una API, qué contiene una petición y qué contiene una respuesta. Y, sobre todo, descubriremos algo que sorprende a casi todo el mundo: **el modelo no recuerda nada de una petición a la siguiente**. Veremos cómo se construye, mensaje a mensaje, la «memoria» de una conversación.

Como en los notebooks anteriores, usaremos Qwen3 en LM Studio, y todas las salidas que verás son respuestas reales del modelo.

## **1. Qué es una API**

Piensa en un restaurante. Tú no entras en la cocina, no sabes dónde están los ingredientes ni cómo funciona el horno. Lees la carta, haces tu pedido al camarero de una forma que él entiende («un menú del día, con el segundo plato sin sal») y, al rato, te trae el plato. La carta te dice **qué puedes pedir**; el camarero **lleva tu pedido y te trae el resultado**; la cocina queda fuera de tu vista.

Una **API** (*Application Programming Interface*, «interfaz de programación de aplicaciones») cumple ese mismo papel entre dos programas. Un programa ofrece unos servicios y publica unas reglas para pedirlos: qué se puede pedir, con qué formato y qué se recibirá a cambio. Otro programa sigue esas reglas, hace su petición y recibe la respuesta, sin necesidad de saber cómo funciona el primero por dentro.

Las usamos constantemente sin darnos cuenta:

- la aplicación del tiempo del móvil pide la previsión a un servicio meteorológico;
- una tienda en línea pide a un banco que autorice un pago;
- un mapa incrustado en una página web pide las imágenes a un servicio de mapas.

Y, cuando una aplicación «tiene inteligencia artificial», casi siempre significa que, por detrás, envía peticiones a la API de un modelo de lenguaje.

<div style="display:inline-block; background-color:#fff3cd; border:1px solid #f0c36d; border-left:6px solid #e0a800; border-radius:6px; padding:12px 16px; margin:12px 0; color:#333;">
<b>Una API es un conjunto de reglas que permite a un programa pedir un servicio a otro: qué se puede pedir, en qué formato y qué se recibe a cambio.</b>
</div>

## **2. La API de un modelo de lenguaje**

<div style="text-align:center">
<img src="https://raw.githubusercontent.com/cayetanoguerra/redes-neuronales/main/notebooks/imgs/api/api-01-peticion-respuesta.svg" alt="Una llamada a la API: el programa envía una petición con el modelo, las instrucciones y la pregunta, y recibe una respuesta con el texto generado y el consumo de tokens" width="85%" style="max-width:100%;height:auto;border:1px solid #dae1e8;border-radius:8px;">
</div>

<p style="text-align:center"><em>Una llamada a la API: el programa envía una petición con el modelo, las instrucciones y la pregunta, y recibe una respuesta con el texto generado y el consumo de tokens</em></p>

Usar un modelo a través de su API consiste en enviar una **petición** a una **dirección** y recibir una **respuesta**. Ambas son textos en formato **JSON**, el formato de datos que ya conocimos con las herramientas: pares de nombre y valor entre llaves.

La **petición** incluye, como mínimo:

- `model`: qué modelo debe responder;
- `input`: lo que le enviamos, por ejemplo una pregunta;
- y, si queremos, `instructions`: indicaciones generales sobre cómo debe responder.

La **respuesta** incluye, entre otras cosas:

- `output`: el texto que ha generado el modelo;
- `usage`: cuántos tokens ha leído (entrada) y cuántos ha escrito (salida);
- `id`: un identificador único de esa respuesta.

La **dirección** indica a qué servidor y a qué servicio se envía la petición. Para LM Studio es `http://localhost:1234/v1/responses`: el servidor está en nuestro propio ordenador (`localhost`), escucha en el puerto `1234` y el servicio se llama `responses`. Para los modelos de OpenAI en la nube, la dirección es otra, y además hay que identificarse con una **clave de API**, como veremos al final.

### Dos interfaces: Chat Completions y Responses

OpenAI ha publicado dos interfaces para conversar con sus modelos, y LM Studio imita las dos:

| | Chat Completions | Responses |
|---|---|---|
| Dirección | `/v1/chat/completions` | `/v1/responses` |
| Lo que enviamos | `messages` | `input` (e `instructions`) |
| Dónde viene el texto generado | `choices[0].message.content` | `output` |
| Antigüedad | La primera, de 2023 | La actual, de 2025 |

La celda del final del notebook anterior usaba **Chat Completions**, que es la más extendida y la que encontrarás en muchos tutoriales. A partir de aquí usaremos **Responses**, que es la que OpenAI recomienda hoy y la que necesitaremos para las herramientas y los agentes. Las ideas son las mismas en las dos.

### Preparar los experimentos

Para ejecutar las celdas necesitas LM Studio abierto con el servidor local activo, el modelo `qwen/qwen3-4b-2507` descargado y la biblioteca `openai` de Python (`pip install openai`). Estas celdas no funcionan en Google Colab, porque Colab no puede conectar con el LM Studio de tu ordenador: descarga el notebook y ábrelo con Jupyter en el mismo equipo.

## **3. Una petición hecha a mano**

Empecemos sin ninguna ayuda, para ver exactamente qué viaja de un programa a otro. La siguiente celda construye la petición como un diccionario de Python, la convierte en JSON, la envía a LM Studio y muestra **la respuesta completa**, tal como llega.

In [1]:
import json
import urllib.request

direccion = "http://localhost:1234/v1/responses"

peticion = {
    "model": "qwen/qwen3-4b-2507",
    "instructions": "Responde en español, en un máximo de dos frases.",
    "input": "¿Qué es una API?",
    "temperature": 0,
}

try:
    solicitud = urllib.request.Request(
        direccion,
        data=json.dumps(peticion).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(solicitud, timeout=120) as respuesta_http:
        datos = json.load(respuesta_http)
    print(json.dumps(datos, indent=2, ensure_ascii=False))
except Exception as error:
    print("No se ha podido conectar con LM Studio.")
    print("Comprueba que está abierto, que el servidor está en marcha y que el modelo está descargado.")
    print("Detalle:", error)

{
  "id": "resp_fee35b8b0675d7d2bf9eb9b109a5b782c46886b79d201068",
  "object": "response",
  "created_at": 1794434051,
  "completed_at": 1794434053,
  "status": "completed",
  "incomplete_details": null,
  "model": "qwen/qwen3-4b-2507",
  "previous_response_id": null,
  "instructions": "Responde en español, en un máximo de dos frases.",
  "output": [
    {
      "id": "msg_wnsosbe38g9sfwf7wi2dmo",
      "type": "message",
      "role": "assistant",
      "status": "completed",
      "content": [
        {
          "type": "output_text",
          "text": "Una API (Interfaz de Programación de Aplicaciones) es un conjunto de reglas y protocolos que permite a diferentes software o servicios comunicarse y compartir datos. Se utiliza para integrar funcionalidades entre aplicaciones, como obtener información o realizar acciones sin necesidad de que el usuario acceda directamente al sistema.",
          "annotations": [],
          "logprobs": []
        }
      ]
    }
  ],
  "error": null,

Parece mucho, pero solo hay que fijarse en unos pocos campos:

- **`output`** contiene lo que ha generado el modelo. Es una lista porque, como veremos en los notebooks de herramientas, un modelo puede devolver varias cosas a la vez. Aquí hay un único elemento de tipo `message`, con `role: "assistant"` (el modelo) y, dentro, el texto.
- **`usage`** cuenta los tokens: el modelo ha leído **32** (las instrucciones y la pregunta) y ha escrito **68**. Este dato será protagonista en el resto del notebook.
- **`id`** identifica esta respuesta. Lo usaremos en el apartado 8.
- **`instructions`**, **`model`** y **`temperature`** repiten lo que enviamos.

El resto son ajustes que no hemos tocado y que conservan su valor por defecto.

Hemos tenido que escribir muchas líneas para algo sencillo. Por eso, en la práctica, nadie hace las peticiones a mano.

## **4. La biblioteca de OpenAI**

La biblioteca `openai` de Python hace por nosotros todo el trabajo anterior: construye el JSON, lo envía, comprueba que no haya errores y convierte la respuesta en un objeto cómodo de usar. Primero creamos un **cliente**, que guarda la dirección del servidor:

In [2]:
from openai import OpenAI

# LM Studio no exige una clave real, pero la biblioteca necesita que escribamos alguna.
cliente = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
MODELO = "qwen/qwen3-4b-2507"

try:
    disponibles = [modelo.id for modelo in cliente.models.list()]
    print("Conexión correcta. Modelos disponibles en LM Studio:")
    for nombre in disponibles:
        print("  -", nombre)
except Exception as error:
    print("No se ha podido conectar con LM Studio.")
    print("Detalle:", error)

Conexión correcta. Modelos disponibles en LM Studio:
  - qwen/qwen3-4b-thinking-2507
  - google/gemma-4-e4b
  - qwen3.5-4b
  - text-embedding-nomic-embed-text-v1.5
  - qwen/qwen3-4b-2507


Y ahora la misma petición del apartado anterior, en una sola instrucción:

In [3]:
respuesta = cliente.responses.create(
    model=MODELO,
    instructions="Responde en español, en un máximo de dos frases.",
    input="¿Qué es una API?",
    temperature=0,
)

print(respuesta.output_text)
print()
print("Tokens de entrada:", respuesta.usage.input_tokens)
print("Tokens de salida: ", respuesta.usage.output_tokens)

Una API (Interfaz de Programación de Aplicaciones) es un conjunto de reglas y protocolos que permite a diferentes software o servicios comunicarse y compartir datos. Se utiliza para integrar funcionalidades entre aplicaciones, como obtener información o realizar acciones sin necesidad de que el usuario acceda directamente al sistema.

Tokens de entrada: 32
Tokens de salida:  68


`respuesta.output_text` reúne el texto generado, sin tener que bucear en la lista `output`, y `respuesta.usage` nos da los tokens. Es exactamente la misma respuesta que antes; solo ha cambiado la comodidad.

A partir de ahora usaremos siempre la biblioteca.

## **5. Las instrucciones**

El campo `instructions` es lo que en LM Studio llamábamos *system prompt*: unas indicaciones generales que el modelo recibe antes del mensaje del usuario y que orientan cómo debe responder. Veamos su efecto haciendo **la misma pregunta con dos instrucciones distintas**:

In [4]:
def preguntar(pregunta, instrucciones):
    """Hace una pregunta al modelo con unas instrucciones y devuelve la respuesta de la API."""
    return cliente.responses.create(model=MODELO, instructions=instrucciones,
                                    input=pregunta, temperature=0)

for instrucciones in ["Eres un maestro de primaria. Explica con un ejemplo cotidiano, en dos frases como máximo.",
                      "Eres un ingeniero informático. Responde con precisión técnica, en dos frases como máximo."]:
    print("INSTRUCCIONES:", instrucciones)
    print(preguntar("¿Qué es una API?", instrucciones).output_text)
    print("-" * 90)

INSTRUCCIONES: Eres un maestro de primaria. Explica con un ejemplo cotidiano, en dos frases como máximo.
Una API es como un mensajero que permite a dos aplicaciones hablar sin necesitar que se vean. Por ejemplo, cuando usas una app de redes sociales y te muestra el clima, la app está usando una API para pedir datos del clima sin tener que hacerlo por sí misma.
------------------------------------------------------------------------------------------
INSTRUCCIONES: Eres un ingeniero informático. Responde con precisión técnica, en dos frases como máximo.
Una API (Interfaz de Programación de Aplicaciones) es un conjunto de protocolos, herramientas y definiciones que permiten a diferentes software interactuar y compartir datos. Permite que una aplicación acceda a servicios o funcionalidades de otra sin necesidad de conocer su código interno.
------------------------------------------------------------------------------------------


El mismo modelo, ante la misma pregunta, ha respondido de dos formas muy distintas: con una comparación cotidiana (un mensajero, la aplicación del tiempo) en el primer caso, y con vocabulario técnico (protocolos, código interno) en el segundo.

Las instrucciones son una de las herramientas más potentes que tenemos al programar con modelos de lenguaje. En ellas se suele indicar el **papel** del modelo, el **público** al que se dirige, el **tono**, la **extensión** o el **formato** de las respuestas.

En muchos ejemplos verás las instrucciones escritas de otra forma: como un primer mensaje con el rol `"system"` o `"developer"` dentro de la lista de mensajes. Es lo mismo con otra presentación.

## **6. La API no tiene memoria**

Vamos ahora con lo más importante del notebook. Imagina que estamos preparando una clase y queremos ideas. Hacemos una primera pregunta:

In [5]:
INSTRUCCIONES = "Eres un asistente para docentes. Responde en español, de forma breve y práctica."

PREGUNTAS = [
    "Estoy preparando una clase sobre el ciclo del agua para alumnos de 12 años. "
    "Propón tres actividades, en una línea cada una.",
    "Desarrolla la segunda en cinco pasos.",
    "Adáptala para que dure solo 15 minutos.",
]

print(preguntar(PREGUNTAS[0], INSTRUCCIONES).output_text)

1. Simulación del ciclo del agua con agua, plástico y cajas para representar evaporación, condensación y precipitación.  
2. Creación de un mapa del ciclo del agua con dibujos y etiquetas en grupo.  
3. Juego de roles: cada alumno representa un elemento del ciclo (por ejemplo, el sol, la nube, la lluvia) y explica su función.


Tres actividades, tal como pedimos. Nos interesa la segunda, el mapa del ciclo del agua, así que hacemos lo que haríamos en cualquier chat: pedir que la desarrolle.

In [6]:
print(preguntar(PREGUNTAS[1], INSTRUCCIONES).output_text)

Claro, aquí tienes un ejemplo de desarrollo de una "segunda en cinco pasos" (asumiendo que se refiere a un proceso o estrategia):

**Desarrollo de la segunda estrategia en cinco pasos:**

1. **Revisión del objetivo**: Revisa el propósito específico que debe cumplir la segunda estrategia en el proceso educativo.

2. **Análisis del contexto**: Evalúa el entorno (clase, estudiantes, recursos) para adaptar la estrategia a las necesidades reales.

3. **Diseño del plan**: Elabora un esquema práctico con actividades, tiempos y responsabilidades asignadas.

4. **Implementación gradual**: Aplica la estrategia en pequeñas fases, monitoreando el avance y ajustando según sea necesario.

5. **Evaluación y retroalimentación**: Revisa los resultados con los estudiantes y docentes, y ajusta el enfoque para futuras aplicaciones.

¿Quieres que se adapte a un área específica (matemáticas, lectura, etc.)?


El modelo no sabe de qué le hablamos. Entiende «la segunda» como «una segunda estrategia» cualquiera y se inventa cinco pasos genéricos que no tienen nada que ver con el ciclo del agua. Incluso nos pregunta, al final, a qué área nos referimos.

¿Por qué? Porque **cada petición a la API es independiente**. El modelo no guarda nada de una petición a la siguiente: cuando recibe «Desarrolla la segunda en cinco pasos», ese texto, junto con las instrucciones, es **todo** lo que tiene. Lo que se dijo en la petición anterior no existe para él.

Esto encaja con lo que sabemos desde el notebook 07: un modelo de lenguaje es una función que recibe un contexto y genera la continuación. Sus parámetros no cambian al usarlo. No hay ningún lugar donde pueda «apuntarse» lo que le dijimos antes.

<div style="display:inline-block; background-color:#fff3cd; border:1px solid #f0c36d; border-left:6px solid #e0a800; border-radius:6px; padding:12px 16px; margin:12px 0; color:#333;">
<b>El modelo no recuerda nada entre una petición y la siguiente. Cada petición debe llevar todo lo que el modelo necesita saber.</b>
</div>

Pero entonces, ¿cómo es posible que ChatGPT o el chat de LM Studio sí recuerden lo que hablamos hace un momento? La respuesta es el tema del siguiente apartado: **no lo recuerda el modelo; lo recuerda la aplicación**.

## **7. Construir el contexto, mensaje a mensaje**

El truco es sencillo. La aplicación guarda una lista con toda la conversación y, **en cada petición, se la envía entera al modelo**: las preguntas anteriores, las respuestas anteriores y, al final, la pregunta nueva. Así, el modelo «ve» toda la conversación cada vez, como si la leyera de nuevo desde el principio.

Vamos a programarlo. La lista `conversacion` empieza vacía. La función `conversar`:

1. añade la pregunta del usuario a la lista;
2. envía **la lista completa** al modelo en el campo `input`;
3. añade la respuesta del modelo a la lista, para que forme parte de las peticiones siguientes.

Para ver bien qué ocurre, antes de cada petición la función muestra **lo que se va a enviar**, mensaje a mensaje.

In [7]:
conversacion = []      # aquí se irá acumulando toda la conversación
registro = []          # tokens de entrada y de salida de cada petición

def rol_y_texto(mensaje):
    """Devuelve el rol y el texto de un mensaje, sea una pregunta nuestra o una respuesta del modelo."""
    if isinstance(mensaje, dict):
        return mensaje["role"], mensaje["content"]
    return mensaje.role, "".join(parte.text for parte in mensaje.content)

def mostrar_contexto():
    """Muestra, abreviados, los mensajes que se van a enviar al modelo."""
    n = len(conversacion)
    print(f"   Se envía{'n' if n > 1 else ''} {n} mensaje{'s' if n > 1 else ''}:")
    for mensaje in conversacion:
        rol, texto = rol_y_texto(mensaje)
        etiqueta = "usuario" if rol == "user" else "modelo "
        texto = texto.replace("\n", " ")
        print(f"     [{etiqueta}] {texto[:75]}{'…' if len(texto) > 75 else ''}")

def conversar(pregunta):
    """Añade la pregunta a la conversación, la envía completa y guarda la respuesta."""
    conversacion.append({"role": "user", "content": pregunta})    # 1. la pregunta nueva

    print(f"PETICIÓN {len(registro) + 1}")
    mostrar_contexto()

    respuesta = cliente.responses.create(model=MODELO, instructions=INSTRUCCIONES,
                                         input=conversacion, temperature=0)   # 2. toda la lista
    conversacion.extend(respuesta.output)                           # 3. la respuesta, a la lista

    registro.append((respuesta.usage.input_tokens, respuesta.usage.output_tokens))
    print(f"   Tokens de entrada: {respuesta.usage.input_tokens}")
    print()
    print("RESPUESTA:")
    print(respuesta.output_text)
    print("=" * 90)

Hacemos las tres preguntas, una detrás de otra:

In [8]:
for pregunta in PREGUNTAS:
    conversar(pregunta)

PETICIÓN 1
   Se envía 1 mensaje:
     [usuario] Estoy preparando una clase sobre el ciclo del agua para alumnos de 12 años.…
   Tokens de entrada: 64

RESPUESTA:
1. Simulación del ciclo del agua con agua, plástico y cajas para representar evaporación, condensación y precipitación.  
2. Creación de un mapa del ciclo del agua con dibujos y etiquetas en grupo.  
3. Juego de roles: cada alumno representa un elemento del ciclo (por ejemplo, el sol, la nube, la lluvia) y explica su función.
PETICIÓN 2
   Se envían 3 mensajes:
     [usuario] Estoy preparando una clase sobre el ciclo del agua para alumnos de 12 años.…
     [modelo ] 1. Simulación del ciclo del agua con agua, plástico y cajas para representa…
     [usuario] Desarrolla la segunda en cinco pasos.
   Tokens de entrada: 176

RESPUESTA:
1. Dividir a los alumnos en grupos pequeños y asignar a cada grupo un elemento del ciclo del agua (por ejemplo, nube, río, sol).  
2. Pedir a cada grupo que dibuje su elemento y cómo interactúa con 

Ahora sí, la conversación tiene sentido. El modelo ha desarrollado el mapa del ciclo del agua, que era la segunda actividad, y después lo ha adaptado a un tiempo más corto.

Fíjate en cómo crece lo que se envía:

- **Petición 1:** un solo mensaje, nuestra pregunta.
- **Petición 2:** tres mensajes: la primera pregunta, la respuesta del modelo con las tres actividades y la pregunta nueva. Gracias a la respuesta anterior, «la segunda» ya significa algo.
- **Petición 3:** cinco mensajes: toda la conversación anterior y la pregunta nueva. «Adáptala» se refiere a la actividad desarrollada en la respuesta anterior.

<div style="text-align:center">
<img src="https://raw.githubusercontent.com/cayetanoguerra/redes-neuronales/main/notebooks/imgs/api/api-02-contexto-crece.svg" alt="Cada petición incluye toda la conversación anterior más la pregunta nueva. La respuesta de cada petición pasa a formar parte de la siguiente" width="85%" style="max-width:100%;height:auto;border:1px solid #dae1e8;border-radius:8px;">
</div>

<p style="text-align:center"><em>Cada petición incluye toda la conversación anterior más la pregunta nueva. La respuesta de cada petición pasa a formar parte de la siguiente</em></p>

Los tokens de entrada cuentan la misma historia:

In [9]:
print(f"{'Petición':>8} {'T. entrada':>11} {'T. salida':>10}")
for numero, (entrada, salida) in enumerate(registro, start=1):
    print(f"{numero:>8} {entrada:>11} {salida:>10}   " + "█" * round(entrada / 10))

Petición  T. entrada  T. salida
       1          64         91   ██████
       2         176        141   ██████████████████
       3         341        127   ██████████████████████████████████


La pregunta de la tercera petición es cortísima («Adáptala para que dure solo 15 minutos»), pero el modelo ha tenido que leer **341 tokens**, más de cinco veces los de la primera. Cada respuesta del modelo se convierte en entrada de todas las peticiones siguientes, así que **el contexto crece con cada turno de la conversación**.

Antes de seguir, un detalle que conviene no pasar por alto. Si sumas los tiempos de la última respuesta (2 + 3 + 1 + 3 + 1) obtienes **10 minutos**, no 15. El modelo ha cumplido el encargo a medias y lo presenta con total naturalidad. Es otro recordatorio de que hay que revisar lo que devuelve, por muy bien redactado que esté.

<div style="display:inline-block; background-color:#fff3cd; border:1px solid #f0c36d; border-left:6px solid #e0a800; border-radius:6px; padding:12px 16px; margin:12px 0; color:#333;">
<b>La memoria de una conversación no está en el modelo, sino en el programa: en cada petición se reenvía toda la conversación anterior. Por eso el contexto, y con él el número de tokens, crece en cada turno.</b>
</div>

Esto explica cosas que quizá hayas observado al usar estos asistentes:

- **Al pulsar «Nueva conversación», el modelo lo «olvida» todo.** Simplemente, la aplicación empieza una lista vacía.
- **En conversaciones muy largas, el asistente se vuelve más lento** y puede perder de vista lo que se dijo al principio, como veremos en el apartado 9.
- **Un error en una respuesta se arrastra.** Si el modelo se equivoca y no lo corregimos, ese error forma parte del contexto de todas las peticiones siguientes.

## **8. Dejar que el servidor guarde la conversación**

Reenviar la conversación completa funciona, pero obliga a nuestro programa a guardarla y a enviarla cada vez. La API Responses ofrece una alternativa: pedir al servidor que **guarde** cada respuesta (`store=True`) y, en la petición siguiente, indicarle **de qué respuesta venimos** con su identificador, `previous_response_id`. Entonces basta con enviar la pregunta nueva: el servidor recupera por su cuenta la conversación anterior.

LM Studio también lo admite. Repitamos las tres preguntas de esta forma:

In [10]:
anterior = None
for numero, pregunta in enumerate(PREGUNTAS, start=1):
    if anterior is None:
        respuesta = cliente.responses.create(model=MODELO, instructions=INSTRUCCIONES,
                                             input=pregunta, store=True, temperature=0)
    else:
        respuesta = cliente.responses.create(model=MODELO, instructions=INSTRUCCIONES,
                                             input=pregunta, previous_response_id=anterior,
                                             temperature=0)
    print(f"Petición {numero}: enviamos solo la pregunta nueva ({len(pregunta)} caracteres)")
    print(f"   id de la respuesta: {respuesta.id[:22]}…   tokens de entrada: {respuesta.usage.input_tokens}")
    anterior = respuesta.id

print()
print("Última respuesta:")
print(respuesta.output_text)

Petición 1: enviamos solo la pregunta nueva (123 caracteres)
   id de la respuesta: resp_fce552057314f8ae0…   tokens de entrada: 64
Petición 2: enviamos solo la pregunta nueva (37 caracteres)
   id de la respuesta: resp_e587f82114f82150e…   tokens de entrada: 176
Petición 3: enviamos solo la pregunta nueva (39 caracteres)
   id de la respuesta: resp_a35e79d1f6085dbf5…   tokens de entrada: 341

Última respuesta:
1. Dividir a los alumnos en grupos de 3 y asignar un elemento del ciclo del agua en 2 minutos.  
2. En 3 minutos, dibujar en papel tamaño A4 su elemento y cómo se conecta con otro.  
3. Un minuto para pegar los dibujos en el pizarrón formando el mapa.  
4. 3 minutos para que cada grupo explique su parte en voz alta.  
5. 1 minuto de cierre: preguntar “¿Qué pasaría sin la lluvia?” y compartir respuestas breves.


Las respuestas son las mismas que en el apartado anterior. Y fíjate en los tokens de entrada: **64, 176 y 341, exactamente los mismos**. Aunque nosotros solo hemos enviado la pregunta nueva, el servidor ha reconstruido la conversación y se la ha pasado entera al modelo. El modelo sigue sin memoria; lo único que ha cambiado es **quién guarda la conversación**: antes, nuestro programa; ahora, el servidor.

Por eso, en los servicios de pago, esta opción no ahorra dinero: la documentación de OpenAI advierte que, aunque se use `previous_response_id`, todos los tokens anteriores de la conversación se cobran como tokens de entrada. Lo que ahorra es trabajo al programador.

OpenAI ofrece además una tercera opción, la **API de conversaciones** (*Conversations*), que guarda en el servidor una conversación con su propio identificador, para retomarla otro día o desde otro dispositivo. La idea de fondo es siempre la misma.

| ¿Quién guarda la conversación? | Qué enviamos en cada petición | Qué recibe el modelo |
|---|---|---|
| Nuestro programa (apartado 7) | Toda la conversación | Toda la conversación |
| El servidor, con `previous_response_id` | Solo la pregunta nueva | Toda la conversación |

## **9. El límite: la ventana de contexto**

Si el contexto crece en cada turno, ¿puede crecer indefinidamente? No. Como vimos en LM Studio, cada modelo tiene una **ventana de contexto**: el número máximo de tokens que puede tener en cuenta a la vez. En la captura del notebook anterior aparecía como «168/8192»: 168 tokens usados de un máximo de 8192.

Cuando una conversación se acerca a ese límite, la aplicación tiene que hacer algo, porque no puede enviar más tokens de los que caben. Las soluciones habituales son:

- **recortar** los mensajes más antiguos, que el modelo deja de ver;
- **resumir** la parte antigua de la conversación y enviar el resumen en su lugar;
- **avisar** al usuario de que conviene empezar una conversación nueva.

En todos los casos se pierde algo de información. Por eso, si una conversación larga cambia de tema, suele dar mejores resultados empezar una nueva: el modelo trabaja con un contexto más corto y más relevante, responde antes y, en un servicio de pago, cuesta menos.

Las ventanas de contexto de los modelos grandes en la nube son mucho mayores que la de nuestro modelo local, con cientos de miles de tokens, pero el principio es el mismo: todo lo que el modelo «sabe» de la conversación es lo que cabe en su contexto.

## **10. La misma API, en la nube**

Todo lo que hemos hecho funciona igual con los modelos de OpenAI en la nube. Solo cambian tres cosas:

| | LM Studio (local) | OpenAI (nube) |
|---|---|---|
| Dirección | `http://localhost:1234/v1` | La de OpenAI, que la biblioteca ya conoce |
| Clave de API | Cualquier texto | Una clave personal y secreta |
| Modelo | `qwen/qwen3-4b-2507` | Uno de los modelos de OpenAI |
| Coste | Gratuito | Se paga por cada token de entrada y de salida |
| Datos | No salen del ordenador | Se envían a los servidores de OpenAI |

La **clave de API** es una contraseña larga que identifica a quien hace la petición, para que el servicio sepa a quién cobrar. Se obtiene en la plataforma para desarrolladores de OpenAI y hay que tratarla como una contraseña: **nunca** debe escribirse dentro de un notebook que se vaya a compartir ni subirse a internet. Lo habitual es guardarla en una **variable de entorno** llamada `OPENAI_API_KEY`, de donde la biblioteca la lee automáticamente.

Con la clave guardada, el código queda así. Esta celda no se ejecuta en el notebook, porque necesita una clave real y cada petición tiene un coste; sustituye `NOMBRE_DEL_MODELO` por uno de los modelos de OpenAI si quieres probarla.

```python
from openai import OpenAI

cliente_nube = OpenAI()     # sin base_url: usa los servidores de OpenAI y la clave de OPENAI_API_KEY

respuesta = cliente_nube.responses.create(
    model="NOMBRE_DEL_MODELO",
    instructions="Responde en español, en un máximo de dos frases.",
    input="¿Qué es una API?",
)
print(respuesta.output_text)
```

Compáralo con el del apartado 4: solo ha cambiado la forma de crear el cliente y el nombre del modelo. Todo lo demás, incluida la manera de construir el contexto de una conversación, es idéntico. Esa es la gran ventaja de que LM Studio imite la API de OpenAI: lo que aprendamos con el modelo local nos servirá con cualquier modelo de la nube.

Recuerda, además, lo que vimos en el notebook anterior sobre la privacidad: con un modelo en la nube, todo lo que enviamos, incluida la conversación completa de cada petición, sale de nuestro ordenador. Si trabajamos con datos del alumnado, conviene tenerlo muy presente.

## **Resumen**

- Una **API** es un conjunto de reglas que permite a un programa pedir un servicio a otro. Usar un modelo de lenguaje desde un programa consiste en enviar una **petición** a su API y recibir una **respuesta**.
- La petición indica el **modelo**, las **instrucciones** y la **entrada**; la respuesta contiene el texto generado (`output`), los **tokens** consumidos (`usage`) y un **identificador** (`id`). Ambas viajan en formato **JSON**.
- La biblioteca `openai` construye y envía las peticiones por nosotros. Con ella, el mismo código sirve para LM Studio y para los modelos de OpenAI en la nube.
- Las **instrucciones** orientan el papel, el tono y el formato de las respuestas.
- **El modelo no tiene memoria.** Cada petición es independiente y debe contener todo lo que el modelo necesita saber.
- Para mantener una conversación, el programa **reenvía en cada petición toda la conversación anterior** más la pregunta nueva. Por eso el contexto crece con cada turno: en nuestro ejemplo, de 64 a 176 y a 341 tokens.
- Con `previous_response_id` la conversación la guarda el servidor, pero el modelo sigue recibiendo el contexto completo.
- El contexto tiene un límite, la **ventana de contexto**. Las conversaciones muy largas se recortan o se resumen, y pierden información.

## **Ejercicios**

1. **Tus instrucciones.** Cambia las instrucciones del apartado 5 para que el modelo responda como un profesor de tu asignatura dirigiéndose a tu alumnado. Compara el resultado con las dos versiones del apartado.
2. **Sin memoria.** Usa la función `preguntar` para enviar primero «Me llamo Ana y doy clase de Física» y después «¿Cómo me llamo?». ¿Qué responde el modelo? Explica por qué.
3. **Con memoria.** Haz lo mismo con la función `conversar`. Antes, vacía la conversación con `conversacion.clear()` y `registro.clear()`. ¿Qué responde ahora? ¿Cuántos mensajes se envían en la segunda petición?
4. **Una conversación más larga.** Continúa la conversación del apartado 7 con dos o tres preguntas más y observa cómo crecen los tokens de entrada. ¿Crecen siempre lo mismo en cada turno? ¿De qué depende?
5. **Corregir al modelo.** En la última respuesta del apartado 7 los tiempos suman 10 minutos. Añade un turno más a la conversación con `conversar` pidiéndole que lo corrija. ¿Lo consigue? ¿Por qué puede corregirlo, si el modelo no tiene memoria?

## **Glosario breve**

**API.** Conjunto de reglas que permite a un programa pedir un servicio a otro: qué se puede pedir, en qué formato y qué se recibe a cambio.

**Petición y respuesta.** Mensajes que intercambian dos programas a través de una API: el que pide envía una petición y el que ofrece el servicio devuelve una respuesta.

**Dirección (*endpoint*).** Dirección de internet a la que se envían las peticiones de un servicio concreto, como `http://localhost:1234/v1/responses`.

**JSON.** Formato de texto para escribir datos estructurados como pares de nombre y valor.

**Clave de API.** Contraseña que identifica a quien hace las peticiones a un servicio de pago. Debe mantenerse en secreto.

**Variable de entorno.** Valor guardado en el sistema operativo, fuera del código, que los programas pueden leer. Es el lugar recomendado para guardar las claves de API.

**Instrucciones (*system prompt*).** Indicaciones generales que recibe el modelo antes de los mensajes del usuario y que orientan cómo debe responder.

**Rol.** Etiqueta que indica quién escribe cada mensaje de una conversación: el usuario (`user`), el modelo (`assistant`) o el programador (`system` o `developer`).

**Contexto.** Todo lo que el modelo recibe en una petición. En una conversación incluye las instrucciones, los mensajes anteriores y la pregunta nueva.

**Ventana de contexto.** Número máximo de tokens que un modelo puede tener en cuenta en una petición.

**`previous_response_id`.** Parámetro de la API Responses que indica al servidor de qué respuesta anterior continúa la conversación, para que la recupere por su cuenta.

[← Notebook 8](https://colab.research.google.com/github/cayetanoguerra/redes-neuronales/blob/main/notebooks/08-modelos-locales.ipynb)